In [ ]:
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "aiobotocore>=2.25.2",
#     "boto3>=1.43.54",
#     "httpx2>=2.9.0",
#     "obstore>=0.11.0",
#     "pyarrow>=22.0.0",
#     "pystac>=1.15.1",
#     "tqdm>=4.67.1",
# ]
# ///

# GEDI Tiled v2
The Tiled GEDI Database is a geoparquet copy of the GEDI dataset that is hive-partitioned by a spatial grid and year. It has been generated using MAAP DPS and now we want to copy (some of) it over to the nasa-maap-datastore bucket and add a collection record to the MAAP STAC.

Later on we may decide to add STAC items (one per tile/year) but for now we are just going to expose the collection-level access pattern via a manifest.txt file that contains all of the S3 URIs for the parquet files.

In [1]:
import asyncio
import io
import json
import os
import re
import time
from datetime import datetime, UTC
from pathlib import Path
from typing import Any, Dict, List, Tuple, TYPE_CHECKING

import boto3
import httpx2 as httpx
import obstore as obs
import pyarrow.parquet as pq
from aiobotocore.session import get_session
from boto3 import Session
from botocore.exceptions import ClientError
from obstore.auth.boto3 import Boto3CredentialProvider
from obstore.store import S3Store
from pystac import Asset, Collection, Extent, Provider, SpatialExtent, TemporalExtent
from pystac.extensions.projection import ProjectionExtension
from pystac.extensions.storage import StorageExtension, StorageScheme
from pystac.extensions.table import TableExtension
from pystac.extensions.version import VersionExtension
from tqdm.asyncio import tqdm

if TYPE_CHECKING:
    from obstore.store import ObjectStore

stage = os.getenv("STAGE", "test")

if stage == "prod":
    STAC_LOADER_SNS_TOPIC_ARN = "arn:aws:sns:us-west-2:916098889494:MAAP-STAC-dev-pgSTAC-stacitemloaderTopicD9D06088-m0iaNFNtnXUP"
else:
    STAC_LOADER_SNS_TOPIC_ARN = "arn:aws:sns:us-west-2:916098889494:MAAP-STAC-test-pgSTAC-stacitemloaderTopicD9D06088-LutBraKgk6sT"

MAAP_OPS_BUCKET = "maap-ops-workspace"
ASSET_DEST_BUCKET = "nasa-maap-data-store" if stage == "prod" else "hrodmn-scratch"

VERSION = "v2"

SOURCE_PREFIX = "shared/ameliah/gedi-test/brazil_tiles/"
DEST_PREFIX = "file-staging/nasa-map/gedi-tiled-{version}/"

AWS_REGION = "us-west-2"

session = Session()
sns_client = boto3.client("sns")
source_store = S3Store(
    bucket=MAAP_OPS_BUCKET,
    prefix=SOURCE_PREFIX,
    region="us-west-2",
    credential_provider=Boto3CredentialProvider(session),
)

dest_store = S3Store(
    bucket=ASSET_DEST_BUCKET,
    prefix=DEST_PREFIX.format(version=VERSION),
    region=AWS_REGION,
    credential_provider=Boto3CredentialProvider(session),
)


async def copy_s3_keys(
    source_bucket_name: str,
    destination_bucket_name: str,
    key_mapping_tuples: List[Tuple[str, str]],
    aws_region: str = AWS_REGION,
    max_concurrent_copies: int = 50,
):
    """
    Copies a specific set of S3 objects from one bucket to another asynchronously,
    allowing renaming in the destination, without local download, and displays
    progress using tqdm.

    Args:
        source_bucket_name (str): The name of the source S3 bucket.
        destination_bucket_name (str): The name of the destination S3 bucket.
        key_mapping_tuples (list): A list of tuples, where each tuple is
                                   (source_key, destination_key).
                                   source_key: The key of the object in the source bucket.
                                   destination_key: The desired key for the object
                                                    in the destination bucket.
        aws_region (str): The AWS region of the buckets (e.g., "us-east-1").
        max_concurrent_copies (int): The maximum number of S3 copy operations
                                     to run concurrently. Adjust based on your
                                     AWS account limits and network conditions.
    """
    session = get_session()

    # Create an S3 client asynchronously within an async context manager
    async with session.create_client("s3", region_name=aws_region) as s3_client:
        # Use an asyncio Semaphore to limit the number of concurrent tasks
        # This prevents overwhelming S3 or hitting your connection limits
        semaphore = asyncio.Semaphore(max_concurrent_copies)

        async def _copy_single_object(src_key, dest_key):
            """Helper async function to copy a single S3 object."""
            async with semaphore:
                try:
                    copy_source = {"Bucket": source_bucket_name, "Key": src_key}
                    await s3_client.copy_object(
                        CopySource=copy_source,
                        Bucket=destination_bucket_name,
                        Key=dest_key,
                    )
                    return True
                except ClientError as e:
                    if e.response["Error"]["Code"] == "NoSuchKey":
                        tqdm.write(
                            f"Error: Source key '{src_key}' not found in bucket '{source_bucket_name}'."
                        )
                    else:
                        tqdm.write(f'Error copying "{src_key}" to "{dest_key}": {e}')
                    return False
                except Exception as e:
                    tqdm.write(
                        f"An unexpected error occurred while copying '{src_key}' to '{dest_key}': {e}"
                    )
                    return False

        tasks = []
        for src_key, dest_key in key_mapping_tuples:
            tasks.append(_copy_single_object(src_key, dest_key))

        results = await tqdm.gather(*tasks, desc="Copying S3 Objects", unit="file")

    successful_copies = results.count(True)
    failed_copies = results.count(False)
    if failed_copies > 0:
        print(
            f"\nCompleted S3 copy operation. {successful_copies} files copied successfully, {failed_copies} failed."
        )
    else:
        print(
            f"\nCompleted S3 copy operation. All {successful_copies} files copied successfully."
        )

In [2]:
keys = []

for prefix in ["data/", "metadata/"]:
    list_stream = source_store.list_async(prefix=prefix)
    async for batch in list_stream:
        for obj in batch:
            keys.append(obj["path"])

print(len(keys))
all(key.endswith(".parquet") for key in keys)

22162


True

In [3]:
# if testing, just use the first 5 and last 5 keys
if stage != "prod":
    keys = keys[:5] + keys[-5:]

source_keys = [SOURCE_PREFIX + key for key in keys]
dest_keys = [DEST_PREFIX.format(version=VERSION) + key for key in keys]

print(f"s3://{ASSET_DEST_BUCKET}/{dest_keys[0]}")

s3://hrodmn-scratch/file-staging/nasa-map/gedi-tiled-v2/data/tile_id=N00_W047/year=2019/data_0.parquet


In [4]:
await copy_s3_keys(
    source_bucket_name=MAAP_OPS_BUCKET,
    destination_bucket_name=ASSET_DEST_BUCKET,
    key_mapping_tuples=zip(source_keys, dest_keys),
    aws_region=AWS_REGION,
)

Copying S3 Objects: 100%|█████████████████████████████████████████████| 10/10 [00:05<00:00,  1.82file/s]


Completed S3 copy operation. All 10 files copied successfully.


## File manifest and SQL bootstrap script
Users that have `s3:ListBucket` privileges on the nasa-maap-data-store bucket could just query this dataset with duckdb's `read_parquet('s3://nasa-maap-data-store/file-staging/gedi-tiled-v2/data/**/*.parquet` but since most users won't have those privileges we can publish a file manifest that users can read from duckdb and pass the contents to their actual `read_parquet` query.

In [5]:
asset_root = f"s3://{ASSET_DEST_BUCKET}/{DEST_PREFIX.format(version=VERSION)}"
data_keys = [key for key in keys if key.startswith("data/")]
manifest_key = DEST_PREFIX.format(version=VERSION) + "manifest.txt"
manifest_href = f"s3://{ASSET_DEST_BUCKET}/{manifest_key}"

manifest_body = "\n".join(f"{asset_root}{key}" for key in sorted(data_keys)) + "\n"
obs.put(dest_store, "manifest.txt", manifest_body.encode("utf-8"))

bootstrap_sql_key = DEST_PREFIX.format(version=VERSION) + f"gedi-tiled-{VERSION}.duckdb.sql"
bootstrap_sql_href = f"s3://{ASSET_DEST_BUCKET}/{bootstrap_sql_key}"
bootstrap_sql = f"""INSTALL httpfs;
LOAD httpfs;

SET VARIABLE gedi_v2_files = (
    SELECT list(href)
    FROM read_csv(
        '{manifest_href}',
        header = false,
        columns = {{'href': 'VARCHAR'}}
    )
);

CREATE OR REPLACE VIEW gedi_tiled_v2 AS
SELECT *
FROM read_parquet(
    getvariable('gedi_v2_files'),
    hive_partitioning = true
);
"""
obs.put(dest_store, f"gedi-tiled-{VERSION}.duckdb.sql", bootstrap_sql.encode("utf-8"))

print(manifest_href)
print(bootstrap_sql_href)

s3://hrodmn-scratch/file-staging/nasa-map/gedi-tiled-v2/manifest.txt
s3://hrodmn-scratch/file-staging/nasa-map/gedi-tiled-v2/gedi-tiled-v2.duckdb.sql


In [6]:
print(bootstrap_sql)

INSTALL httpfs;
LOAD httpfs;

SET VARIABLE gedi_v2_files = (
    SELECT list(href)
    FROM read_csv(
        's3://hrodmn-scratch/file-staging/nasa-map/gedi-tiled-v2/manifest.txt',
        header = false,
        columns = {'href': 'VARCHAR'}
    )
);

CREATE OR REPLACE VIEW gedi_tiled_v2 AS
SELECT *
FROM read_parquet(
    getvariable('gedi_v2_files'),
    hive_partitioning = true
);



## Schema metadata

Generate a JSON schema from one sample GeoParquet file and upload it beside the data so the STAC collection can reference it.

In [7]:
data_sample_key = next(key for key in data_keys)
schema_key = DEST_PREFIX.format(version=VERSION) + "schema.json"

obj = dest_store.get(data_sample_key)
schema = pq.read_schema(io.BytesIO(obj.bytes()))
geo_metadata = json.loads(schema.metadata.get(b"geo", b"{}")) if schema.metadata else {}
for column in geo_metadata.get("columns", {}).values():
    column.pop("bbox", None)

schema_doc = {
    "title": "Tiled GEDI GeoParquet schema",
    "version": VERSION,
    "sample": f"s3://{ASSET_DEST_BUCKET}{DEST_PREFIX}{data_sample_key}",
    "format": "GeoParquet",
    "partition_columns": [
        {"name": "tile_id", "type": "string", "description": "1° tile identifier, e.g. N00_W047"},
        {"name": "year", "type": "integer", "description": "UTC year of absolute_time"},
    ],
    "geo": geo_metadata,
    "source_products": ["GEDI L2A", "GEDI L2B", "GEDI L4A", "GEDI L4C"],
    "profile_columns": {
        "rh": "Expanded to rh_0 through rh_100 from the GEDI L2A rh profile.",
        "cover_z": "Expanded to cover_z_0 through cover_z_29 from GEDI L2B.",
        "pai_z": "Expanded to pai_z_0 through pai_z_29 from GEDI L2B.",
        "pavd_z": "Expanded to pavd_z_0 through pavd_z_29 from GEDI L2B.",
        "xvar": "Expanded from GEDI L4A xvar profile columns when present.",
    },
    "columns": [
        {
            "name": field.name,
            "type": str(field.type),
            "nullable": field.nullable,
        }
        for field in schema
    ],
}

obs.put(
    dest_store,
    "schema.json",
    json.dumps(schema_doc, indent=2).encode("utf-8")
)

schema_href = f"s3://{ASSET_DEST_BUCKET}/{schema_key}"
print(schema_href)

s3://hrodmn-scratch/file-staging/nasa-map/gedi-tiled-v2/schema.json


## STAC collection



In [8]:
# define the spatial and temporal extent using the parquet asset keys
minx, miny = float("inf"), float("inf")
maxx, maxy = float("-inf"), float("-inf")
minyear, maxyear = float("inf"), float("-inf")
tile_ids = set()
years = set()

data_keys = [key for key in keys if key.startswith("data/")]

for key in data_keys:
    match = re.search(
        r"tile_id=(?P<tile_id>(?P<lat_dir>[NS])(?P<lat_val>\d+)_(?P<lon_dir>[EW])(?P<lon_val>\d+)).*?year=(?P<year>\d{4})",
        key,
    )
    if not match:
        continue

    d = match.groupdict()
    tile_ids.add(d["tile_id"])

    # Tile IDs encode the north/top edge of each 1-degree tile.
    lat = float(d["lat_val"]) * (-1 if d["lat_dir"] == "S" else 1)
    lon = float(d["lon_val"]) * (-1 if d["lon_dir"] == "W" else 1)

    minx = min(minx, lon)
    miny = min(miny, lat - 1.0)
    maxx = max(maxx, lon + 1.0)
    maxy = max(maxy, lat)

    year = int(d["year"])
    years.add(year)
    minyear = min(minyear, year)
    maxyear = max(maxyear, year)

temporal_start = datetime(minyear, 1, 1, tzinfo=UTC)
temporal_end = datetime(maxyear, 12, 31, 23, 59, 59, tzinfo=UTC)
collection_bbox = [minx, miny, maxx, maxy]

collection = Collection(
    id="gedi-tiled-v2",
    description=(
        "A tiled GeoParquet copy of selected GEDI footprint-level products, "
        "partitioned by 1° spatial tile and observation year for efficient cloud querying. "
        "Each row represents a GEDI shot/footprint, joined across GEDI L2A, L2B, "
        "L4A, and L4C where available, with point geometry in EPSG:4326. "
        "Recommended access pattern: run the SQL code in the gedi-tiled-v2.duckdb.sql asset "
        "to create a DuckDB view named gedi_tiled_v2, then query that view. The bootstrap "
        "SQL reads manifest.txt, a newline-delimited list of Parquet S3 URIs, so users can "
        "query the collection without discovering files themselves."
    ),
    extent=Extent(
        spatial=SpatialExtent(bboxes=[collection_bbox]),
        temporal=TemporalExtent(intervals=[[temporal_start, temporal_end]]),
    ),
    license="various",
    keywords=[
        "GEDI",
        "LiDAR",
        "biomass",
        "forest structure",
        "aboveground biomass",
        "canopy height",
        "GeoParquet",
        "MAAP",
    ],
    providers=[
        Provider(
            name="NASA/GSFC/GEDI",
            roles=["producer", "licensor"],
            url="https://science.nasa.gov/mission/gedi/",
        ),
        Provider(
            name="MAAP",
            roles=["processor", "host"],
            url="https://maap-project.org/",
        ),
        Provider(
            name="gedi_tiler",
            roles=["processor"],
            url="https://github.com/ameliaholcomb/gedi_tiler",
        ),
    ],
)

VersionExtension.ext(collection, add_if_missing=True).version = VERSION
ProjectionExtension.summaries(collection, add_if_missing=True).epsg = [4326]

table_ext = TableExtension.ext(collection, add_if_missing=True)
table_ext.primary_geometry = "geometry"
table_ext.properties["table:tables"] = {
    "gedi_footprints": {
        "name": "gedi_footprints",
        "description": (
            "Footprint-level GEDI measurements stored as GeoParquet, "
            "with one row per GEDI shot. Column-level schema is provided "
            "by the schema asset."
        ),
    }
}

StorageExtension.ext(collection, add_if_missing=True).apply(
    schemes={
        "aws-s3": StorageScheme.create(
            type="aws-s3",
            platform="https://{bucket}.s3.{region}.amazonaws.com",
            region=AWS_REGION,
            bucket=ASSET_DEST_BUCKET,
        )
    }
)

asset_refs = ["aws-s3"]

data_asset = Asset(
    href=asset_root + "data/",
    media_type="application/vnd.apache.parquet",
    roles=["data"],
    title="Hive-partitioned GEDI GeoParquet data",
    description=(
        "Parquet files partitioned by the cube dimensions tile_id and year. "
        "Paths follow data/tile_id=<tile_id>/year=<year>/data_*.parquet."
    ),
)
collection.add_asset("data-root", data_asset)
StorageExtension.ext(data_asset).refs = asset_refs

manifest_asset = Asset(
    href=manifest_href,
    media_type="text/plain",
    roles=["metadata"],
    title="Parquet file manifest",
    description=(
        "Newline-delimited S3 URIs for every GEDI GeoParquet data file. "
        "Used by the DuckDB bootstrap SQL to build the gedi_tiled view."
    ),
)
collection.add_asset("manifest", manifest_asset)
StorageExtension.ext(manifest_asset).refs = asset_refs

bootstrap_sql_asset = Asset(
    href=bootstrap_sql_href,
    media_type="application/sql",
    roles=["metadata"],
    title="DuckDB bootstrap SQL",
    description="Creates a gedi_tiled DuckDB view from the manifest asset.",
)
collection.add_asset("duckdb-bootstrap", bootstrap_sql_asset)
StorageExtension.ext(bootstrap_sql_asset).refs = asset_refs

metadata_asset = Asset(
    href=asset_root + "metadata/",
    media_type="application/vnd.apache.parquet",
    roles=["metadata"],
    title="Tile/granule planning metadata",
    description="Parquet metadata used to plan and build each GEDI tile.",
)
collection.add_asset("metadata-root", metadata_asset)
StorageExtension.ext(metadata_asset).refs = asset_refs

schema_asset = Asset(
    href=schema_href,
    media_type="application/schema+json",
    roles=["metadata", "schema"],
    title="Parquet schema",
    description="Generated schema for the tiled GEDI GeoParquet files.",
)
collection.add_asset("schema", schema_asset)
StorageExtension.ext(schema_asset).refs = asset_refs

collection.validate()
print(json.dumps(collection.to_dict(), indent=2))




{
  "type": "Collection",
  "id": "gedi-tiled-v2",
  "stac_version": "1.1.0",
  "description": "A tiled GeoParquet copy of selected GEDI footprint-level products, partitioned by 1\u00b0 spatial tile and observation year for efficient cloud querying. Each row represents a GEDI shot/footprint, joined across GEDI L2A, L2B, L4A, and L4C where available, with point geometry in EPSG:4326. Recommended access pattern: run the SQL code in the gedi-tiled-v2.duckdb.sql asset to create a DuckDB view named gedi_tiled_v2, then query that view. The bootstrap SQL reads manifest.txt, a newline-delimited list of Parquet S3 URIs, so users can query the collection without discovering files themselves.",
  "links": [],
  "stac_extensions": [
    "https://stac-extensions.github.io/version/v1.2.0/schema.json",
    "https://stac-extensions.github.io/projection/v2.0.0/schema.json",
    "https://stac-extensions.github.io/table/v1.2.0/schema.json",
    "https://stac-extensions.github.io/storage/v2.0.0/schema.jso

In [9]:
# post collection to StacLoader
response = sns_client.publish(
    TopicArn=STAC_LOADER_SNS_TOPIC_ARN, Message=json.dumps(collection.to_dict())
)
print(response)

{'MessageId': '7d8aa255-84c9-535a-bf37-c0881dd2373a', 'ResponseMetadata': {'RequestId': '52a3a283-0d9f-5763-b272-c05e94843303', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '52a3a283-0d9f-5763-b272-c05e94843303', 'date': 'Fri, 24 Jul 2026 16:46:19 GMT', 'content-type': 'text/xml', 'content-length': '294', 'connection': 'keep-alive'}, 'RetryAttempts': 0}}
